# 📚 Regresión Lineal con Datos de Venezuela
### Notebook para principiantes — Ciencias Sociales y Economía

---

## ¿Qué vamos a hacer?

Vamos a responder esta pregunta:

> **¿Cuándo el gobierno invierte más en educación, ¿más jóvenes van a la universidad?**

Para responderla usaremos datos reales de Venezuela (2000–2023) y aprenderemos a:

1. Cargar y explorar los datos
2. Limpiar los datos
3. Calcular estadísticas básicas
4. Ver cómo se relacionan las variables
5. Crear un modelo de regresión lineal
6. Hacer predicciones

---

### Variables que usaremos

- **X (variable independiente):** Gasto Público en Educación (% del PIB)  
- **Y (variable dependiente):** Tasa de Matrícula Universitaria (%)

> 💡 **¿Qué significa % del PIB?**  
> Es la parte del dinero total que produce el país que el gobierno destina a educación.
> Si el PIB es 100 y el gasto es 5%, el gobierno gasta 5 de cada 100 en educación.

## Paso 1 — Importar librerías

Las librerías son herramientas que nos dan funciones ya hechas para trabajar con datos y gráficos.

In [ ]:
import pandas as pd               # Para trabajar con tablas de datos
import numpy as np                # Para cálculos matemáticos
import matplotlib.pyplot as plt   # Para crear gráficos
import seaborn as sns             # Para gráficos más visuales

from sklearn.linear_model import LinearRegression  # Modelo de regresión
from sklearn.metrics import r2_score               # Métrica de calidad del modelo

print('✅ Librerías cargadas correctamente')

## Paso 2 — Cargar los datos

Creamos el dataset directamente en Python usando un diccionario.  
Cada llave es el nombre de una columna y su valor es la lista de datos.

**Fuentes:** UNESCO, CEPAL, Banco Mundial, INE Venezuela

In [ ]:
datos = {
    'Año':                            [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009,
                                       2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019,
                                       2020, 2021, 2022, 2023],

    'Gasto_Educacion_%PIB':           [4.1, 4.3, 4.0, 3.8, 4.5, 5.1, 5.6, 6.2, 6.0, 5.8,
                                       5.5, 5.7, 6.1, 5.9, 5.4, 4.9, 4.1, 3.5, 2.8, 2.1,
                                       1.9, 2.0, 2.2, 2.3],

    'Matricula_Universitaria_%':      [26.4, 27.1, 27.8, 28.5, 30.2, 33.4, 36.8, 39.1, 40.3, 41.2,
                                       42.0, 43.1, 44.5, 44.8, 44.2, 43.1, 40.8, 37.5, 33.2, 29.4,
                                       26.8, 25.1, 24.3, 23.8],

    'PIB_per_Capita_USD':             [4731, 5044, 4237, 3587, 4799, 6063, 7342, 8335, 9459, 8613,
                                       9067, 10526, 11490, 10170, 9215, 7737, 5941, 4038, 2548, 1651,
                                       1424, 1589, 1867, 2067],

    'Tasa_Desempleo_%':               [13.9, 13.3, 15.8, 18.0, 15.3, 12.4, 10.0, 8.5, 7.3, 7.9,
                                        8.5,  8.2,  7.8,  7.5,  7.2,  7.4,  7.3, 7.0, 6.8, 7.0,
                                        7.2,  7.5,  5.5,  5.2],

    'Becas_Otorgadas_Miles':          [18.2, 19.5, 17.8, 16.1, 22.4, 28.7, 34.2, 39.8, 42.1, 40.3,
                                       38.9, 41.5, 44.8, 43.2, 41.7, 38.4, 31.2, 25.6, 18.9, 13.4,
                                       11.2, 10.8, 11.5, 12.1]
}

df = pd.DataFrame(datos)

print(f'Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas')
print()
df

## Paso 3 — Explorar los datos

Antes de analizar, necesitamos conocer qué tenemos: tipos de datos, si hay vacíos, cuántas filas hay, etc.

In [ ]:
# Muestra las primeras 5 filas
print('--- Primeras 5 filas ---')
df.head()

In [ ]:
# Muestra las últimas 5 filas
print('--- Últimas 5 filas ---')
df.tail()

In [ ]:
# Información general: tipo de cada columna y si hay valores nulos
df.info()

In [ ]:
# Cuenta cuántos valores nulos (vacíos) hay en cada columna
print('Valores nulos por columna:')
print(df.isnull().sum())

In [ ]:
# Cuenta cuántas filas están repetidas
print(f'Filas duplicadas: {df.duplicated().sum()}')

## Paso 4 — Limpiar los datos

Primero introducimos errores a propósito para practicar las herramientas de limpieza.  
Luego las corregimos paso a paso.

| Función | ¿Para qué sirve? |
|---|---|
| `replace()` | Reemplaza un valor específico por otro |
| `dropna()` | Elimina filas que tienen valores vacíos |
| `drop_duplicates()` | Elimina filas repetidas |
| `sort_values()` | Ordena las filas por una columna |

In [ ]:
# Creamos una copia del DataFrame con errores intencionales
df_sucio = df.copy()

df_sucio.loc[4,  'Gasto_Educacion_%PIB']      = None   # Valor vacío (nulo)
df_sucio.loc[11, 'Matricula_Universitaria_%']  = -999   # Código de error
df_sucio.loc[17, 'PIB_per_Capita_USD']         = -999   # Código de error

# Añadimos una fila duplicada (copiamos la primera fila)
df_sucio = pd.concat([df_sucio, df_sucio.iloc[[0]]], ignore_index=True)

# Desordenamos las filas
df_sucio = df_sucio.sample(frac=1, random_state=1).reset_index(drop=True)

print('Dataset con errores creado:')
print(f'  Filas totales  : {len(df_sucio)}')
print(f'  Valores nulos  : {df_sucio.isnull().sum().sum()}')
print(f'  Duplicados     : {df_sucio.duplicated().sum()}')
print(f'  Valores -999   : {(df_sucio == -999).sum().sum()}')

In [ ]:
# LIMPIEZA — Paso a paso

df_limpio = df_sucio.copy()

# 1. replace() — El valor -999 es un código de error, lo convertimos a NaN (vacío)
df_limpio = df_limpio.replace(-999, None)
print(f'Después de replace():        nulos = {df_limpio.isnull().sum().sum()}')

# 2. dropna() — Eliminamos las filas que tienen algún valor vacío
df_limpio = df_limpio.dropna()
print(f'Después de dropna():         filas = {len(df_limpio)}')

# 3. drop_duplicates() — Eliminamos filas repetidas
df_limpio = df_limpio.drop_duplicates()
print(f'Después de drop_duplicates(): filas = {len(df_limpio)}')

# 4. sort_values() — Ordenamos por año de forma ascendente
df_limpio = df_limpio.sort_values('Año').reset_index(drop=True)
print(f'Después de sort_values():    ordenado por Año ✅')

print()
print('Dataset limpio:')
df_limpio

## Paso 5 — Análisis estadístico

### Medidas de tendencia central
Nos dicen dónde se concentran los datos:
- **Media:** el promedio
- **Mediana:** el valor del medio cuando los datos están ordenados
- **Moda:** el valor que más se repite

### Medidas de dispersión
Nos dicen qué tan separados están los datos entre sí:
- **Desviación estándar:** qué tan lejos están los datos de la media
- **Varianza:** la desviación estándar al cuadrado
- **Rango:** la diferencia entre el máximo y el mínimo

In [ ]:
# Estadísticas de Gasto en Educación
gasto = df_limpio['Gasto_Educacion_%PIB']

print('📌 Gasto Público en Educación (% PIB)')
print(f'  Media             : {gasto.mean():.2f}%')
print(f'  Mediana           : {gasto.median():.2f}%')
print(f'  Moda              : {gasto.mode()[0]:.2f}%')
print(f'  Desviación estándar: {gasto.std():.2f}%')
print(f'  Varianza          : {gasto.var():.2f}')
print(f'  Mínimo            : {gasto.min():.2f}%')
print(f'  Máximo            : {gasto.max():.2f}%')
print(f'  Rango             : {gasto.max() - gasto.min():.2f}%')

In [ ]:
# Estadísticas de Matrícula Universitaria
matricula = df_limpio['Matricula_Universitaria_%']

print('📌 Tasa de Matrícula Universitaria (%)')
print(f'  Media             : {matricula.mean():.2f}%')
print(f'  Mediana           : {matricula.median():.2f}%')
print(f'  Moda              : {matricula.mode()[0]:.2f}%')
print(f'  Desviación estándar: {matricula.std():.2f}%')
print(f'  Varianza          : {matricula.var():.2f}')
print(f'  Mínimo            : {matricula.min():.2f}%')
print(f'  Máximo            : {matricula.max():.2f}%')
print(f'  Rango             : {matricula.max() - matricula.min():.2f}%')

In [ ]:
# describe() genera un resumen rápido de todas las columnas numéricas
print('Resumen estadístico completo:')
df_limpio.describe().round(2)

## Paso 6 — Matriz de correlación

La correlación de Pearson **r** nos dice qué tan relacionadas están dos variables.

| Valor de r | Interpretación |
|---|---|
| Cercano a **+1** | Relación positiva fuerte (cuando una sube, la otra también) |
| Cercano a **-1** | Relación negativa fuerte (cuando una sube, la otra baja) |
| Cercano a **0**  | No hay relación lineal |

> 💡 En el mapa de calor: los colores **verdes** indican correlación positiva y los **rojos** negativa.

In [ ]:
# Calculamos la matriz de correlación de Pearson
correlacion = df_limpio.drop(columns='Año').corr(method='pearson').round(2)

# Graficamos el mapa de calor
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlacion,
    annot=True,          # Muestra los números dentro de cada celda
    fmt='.2f',           # Dos decimales
    cmap='RdYlGn',       # Rojo = negativo, Verde = positivo
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 11}
)

plt.title('Matriz de Correlación de Pearson (r)\nIndicadores Educativos — Venezuela 2000–2023',
          fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Mostramos el valor clave
r = correlacion.loc['Matricula_Universitaria_%', 'Gasto_Educacion_%PIB']
print(f'Correlación entre Gasto en Educación y Matrícula Universitaria: r = {r}')
print('→ Correlación positiva muy fuerte ✅')

## Paso 7 — Regresión Lineal

### ¿Qué es la regresión lineal?

Es un método que busca la **línea recta** que mejor describe la relación entre X e Y.

La ecuación de esa recta es:

$$\hat{Y} = \beta_0 + \beta_1 \cdot X$$

- $\hat{Y}$ → el valor que predice el modelo
- $X$ → el valor que conocemos (Gasto en Educación)
- $\beta_0$ → **intercepto**: dónde corta la recta al eje Y (cuando X = 0)
- $\beta_1$ → **pendiente**: cuánto sube Y por cada unidad que sube X

### ¿Cómo se calculan β₀ y β₁?

$$\beta_1 = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sum(x_i - \bar{x})^2}$$

$$\beta_0 = \bar{y} - \beta_1 \cdot \bar{x}$$

Donde $\bar{x}$ es la media de X e $\bar{y}$ es la media de Y.

In [ ]:
# Extraemos los valores de X e Y
X = df_limpio['Gasto_Educacion_%PIB'].values
Y = df_limpio['Matricula_Universitaria_%'].values

# Calculamos las medias
x_media = X.mean()
y_media = Y.mean()

# Calculamos β₁ con la fórmula matemática
numerador   = np.sum((X - x_media) * (Y - y_media))
denominador = np.sum((X - x_media) ** 2)
b1 = numerador / denominador

# Calculamos β₀
b0 = y_media - b1 * x_media

print('Cálculo manual de los coeficientes:')
print(f'  Media de X : {x_media:.4f}')
print(f'  Media de Y : {y_media:.4f}')
print()
print(f'  β₁ (pendiente)   = {b1:.4f}')
print(f'  β₀ (intercepto)  = {b0:.4f}')
print()
print(f'Ecuación de la recta:')
print(f'  Ŷ = {b0:.4f} + {b1:.4f} · X')
print()
print(f'Interpretación:')
print(f'  Por cada 1% adicional del PIB invertido en educación,')
print(f'  la matrícula universitaria aumenta {b1:.2f} puntos porcentuales.')

In [ ]:
# Verificamos con scikit-learn (debe dar el mismo resultado)
X_modelo = X.reshape(-1, 1)   # sklearn requiere que X sea una matriz columna

modelo = LinearRegression()
modelo.fit(X_modelo, Y)

b0_sk = modelo.intercept_
b1_sk = modelo.coef_[0]
Y_pred = modelo.predict(X_modelo)
r2 = r2_score(Y, Y_pred)

print('Resultado con scikit-learn:')
print(f'  β₁ (pendiente)   = {b1_sk:.4f}  ← igual al cálculo manual ✓')
print(f'  β₀ (intercepto)  = {b0_sk:.4f}  ← igual al cálculo manual ✓')
print()
print(f'  R² = {r2:.4f}')
print()
print(f'  El modelo explica el {r2*100:.1f}% de la variabilidad de la matrícula universitaria.')
print(f'  Un R² cercano a 1 indica que el modelo tiene muy buen ajuste.')

## Paso 8 — Gráfico del modelo de regresión

El gráfico muestra:
- 🔵 **Puntos azules:** los datos reales de cada año
- 🔴 **Línea roja:** la recta del modelo de regresión
- ⬛ **Líneas grises:** la distancia entre cada dato real y la recta (llamadas **residuos**)

> El modelo busca minimizar esas distancias grises. A distancias más cortas, mejor ajuste.

In [ ]:
plt.figure(figsize=(10, 7))

# Líneas grises de residuos (distancia entre dato real y predicción)
plt.vlines(X, Y, Y_pred, colors='gray', linewidth=1, alpha=0.5, label='Residuos (distancia)')

# Datos reales
plt.scatter(X, Y, color='steelblue', s=90, zorder=5, label='Datos reales')

# Recta de regresión
plt.plot(X, Y_pred, color='red', linewidth=2.5, label=f'Recta: Ŷ = {b0_sk:.2f} + {b1_sk:.2f}·X')

plt.xlabel('Gasto Público en Educación (% del PIB)', fontsize=12)
plt.ylabel('Tasa de Matrícula Universitaria (%)', fontsize=12)
plt.title('Modelo de Regresión Lineal\nVenezuela 2000–2023', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Cuadro con la ecuación y el R²
texto = f'Ŷ = {b0_sk:.3f} + {b1_sk:.3f}·X\nR² = {r2:.4f}'
plt.text(0.04, 0.96, texto, transform=plt.gca().transAxes,
         fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

## Paso 9 — Predicciones con el modelo

Ahora usamos la ecuación para predecir la matrícula universitaria según distintos niveles de gasto en educación.

La fórmula es simplemente:

$$\hat{Y} = \beta_0 + \beta_1 \cdot X$$

In [ ]:
# Escenarios de predicción
gastos = [1.5, 2.0, 2.5, 3.5, 4.5, 5.5, 6.2, 7.0]
descripciones = [
    'Crisis severa',
    'Nivel actual (2023)',
    'Recuperación leve',
    'Recuperación moderada',
    'Nivel pre-crisis (2004)',
    'Nivel histórico (2009)',
    'Máximo histórico (2007)',
    'Escenario optimista'
]

print('Predicciones del modelo:')
print(f'{"Gasto % PIB":>12} | {"Matrícula predicha":>20} | Escenario')
print('-' * 60)

predicciones = []
for g, desc in zip(gastos, descripciones):
    pred = b0_sk + b1_sk * g
    predicciones.append(pred)
    print(f'{g:>12.1f}% | {pred:>18.2f}%  | {desc}')

In [ ]:
# Gráfico de predicciones
plt.figure(figsize=(10, 6))

# Recta del modelo
x_recta = np.linspace(1.0, 7.5, 100)
y_recta = b0_sk + b1_sk * x_recta
plt.plot(x_recta, y_recta, 'r-', linewidth=2, alpha=0.7, label='Recta del modelo')

# Datos reales
plt.scatter(X, Y, color='steelblue', s=80, zorder=5, label='Datos reales (2000–2023)')

# Predicciones
plt.scatter(gastos, predicciones, color='green', s=130, marker='D',
            zorder=6, label='Predicciones por escenario')

plt.xlabel('Gasto Público en Educación (% del PIB)', fontsize=12)
plt.ylabel('Tasa de Matrícula Universitaria (%)', fontsize=12)
plt.title('Predicciones del Modelo de Regresión Lineal', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

texto = f'Ŷ = {b0_sk:.3f} + {b1_sk:.3f}·X\nR² = {r2:.4f}'
plt.text(0.04, 0.96, texto, transform=plt.gca().transAxes,
         fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

## Paso 10 — Ejercicios propuestos

Completa cada celda reemplazando los `???` con el código correcto.

In [ ]:
# EJERCICIO 1
# El gobierno planea invertir el 4.0% del PIB en educación.
# ¿Qué tasa de matrícula universitaria predice el modelo?

gasto_nuevo = ???
prediccion = b0_sk + b1_sk * gasto_nuevo
print(f'Con un gasto del {gasto_nuevo}% del PIB, la matrícula esperada es: {prediccion:.2f}%')

In [ ]:
# EJERCICIO 2
# Se quiere alcanzar una matrícula universitaria del 45%.
# ¿Cuánto se necesita invertir en educación (% PIB)?
# Pista: despeja X de la ecuación → X = (Y - β₀) / β₁

matricula_objetivo = 45.0
gasto_necesario = (??? - b0_sk) / b1_sk
print(f'Para lograr {matricula_objetivo}% de matrícula se necesita un gasto del {gasto_necesario:.2f}% del PIB')

In [ ]:
# EJERCICIO 3
# Calcula la media y la desviación estándar de la columna 'PIB_per_Capita_USD'
# Usa las funciones .mean() y .std()

media_pib = df_limpio['???'].???()
desv_pib  = df_limpio['???'].???()

print(f'Media del PIB per cápita    : {media_pib:.2f} USD')
print(f'Desviación estándar del PIB : {desv_pib:.2f} USD')

In [ ]:
# EJERCICIO 4
# ¿Cuál fue el año con la mayor tasa de matrícula universitaria?
# Pista: usa .idxmax() para obtener el índice del valor máximo
#        luego usa ese índice para obtener el año con df_limpio.loc[]

indice_max = df_limpio['Matricula_Universitaria_%'].???()
anio_max   = df_limpio.loc[indice_max, '???']
valor_max  = df_limpio.loc[indice_max, 'Matricula_Universitaria_%']

print(f'Año con mayor matrícula: {anio_max} con {valor_max}%')

## Conclusiones

### ¿Qué aprendimos con este análisis?

**1. Sobre los datos de Venezuela**  
El dataset cubre 24 años (2000–2023) y muestra dos fases claras: una etapa de crecimiento hasta 2012–2013, y una caída profunda desde 2016 asociada a la crisis económica.

**2. Sobre la limpieza de datos**  
Antes de analizar cualquier dataset, es necesario revisar si hay valores nulos, errores o duplicados. Las funciones `replace()`, `dropna()`, `drop_duplicates()` y `sort_values()` son las herramientas básicas para dejarlo listo.

**3. Sobre las estadísticas**  
- La media del gasto en educación fue de ~4.3% del PIB, con una caída hasta 1.9% en 2020
- La matrícula universitaria promedió ~35%, llegando a un máximo de 44.8% en 2013
- La alta desviación estándar en ambas variables muestra que hubo cambios muy grandes en el período

**4. Sobre la correlación**  
La correlación de Pearson entre el gasto en educación y la matrícula universitaria es **muy cercana a +1**, lo que confirma que estas dos variables se mueven juntas: cuando el estado invierte más, más jóvenes acceden a la universidad.

**5. Sobre el modelo de regresión lineal**  
- La ecuación encontrada fue: **Ŷ = β₀ + β₁·X**
- El **R²** obtenido es mayor a 0.95, lo que significa que el modelo explica más del 95% del comportamiento de la matrícula universitaria solo con el gasto en educación
- Esto indica que el modelo tiene un **excelente poder predictivo** para este dataset

**6. Reflexión final**  
Los números cuentan historias. En este caso, la regresión lineal nos permite cuantificar de forma precisa algo que intuitivamente tiene sentido: invertir en educación abre puertas. La herramienta matemática convierte esa intuición en una ecuación que podemos usar para tomar decisiones de política pública.